##### Importação dos Pacotes

In [0]:
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import monotonically_increasing_id, current_timestamp, date_format
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

##### Diretórios Catalog

In [0]:
path_bronze          = '/Volumes/01_bronze/schemas/tse/perfil_eleitorado'
path_silver          = '/Volumes/02_silver/schemas/tse/perfil_eleitorado'
path_gold_dimensao   = '/Volumes/03_gold/schemas/dimensao'

##### Consolida os arquivos da camada Silver (fato)

In [0]:
# Consolida arquivos na camada Silver (fato) em um único Dataframe
df_pel = spark.read.format("delta").load(path_silver)


In [0]:
# Validação dos esquemas das colunas no DataFrame
#df_pel.printSchema()

##### Salvar arquivo: tabela Temporária

In [0]:
# Salva o DataFrame em uma tabela Temporária (para leitura SQL)
df_pel.createOrReplaceTempView("df_pel")

In [0]:
%sql
-- Validação dos campos da dimensão
--SELECT DISTINCT
--    CD_GRAU_ESCOLARIDADE,
--    DS_GRAU_ESCOLARIDADE
--FROM df_pel

##### Criação da Dimensão: Estado Civil

In [0]:
# Criação de DataFrame da query para dimensão: Grau Escolaridade
df_grau_escolaridade = spark.sql("""
                SELECT DISTINCT
                CD_GRAU_ESCOLARIDADE AS SkGrauEscolaridade,
                CASE
                    WHEN DS_GRAU_ESCOLARIDADE IN ('LÊ E ESCREVE') THEN ('Le e Escreve')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('ENSINO FUNDAMENTAL INCOMPLETO') THEN ('Ensino Fundamental Incompleto')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('ENSINO FUNDAMENTAL COMPLETO') THEN ('Ensino Fundamental Completo')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('ENSINO MÉDIO INCOMPLETO') THEN ('Ensino Medio Incompleto')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('ENSINO MÉDIO COMPLETO') THEN ('Ensino Medio Completo')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('SUPERIOR INCOMPLETO') THEN ('Ensino Superior Incompleto')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('SUPERIOR COMPLETO') THEN ('Ensino Superior Completo')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('ANALFABETO') THEN ('Analfabeto')
                    WHEN DS_GRAU_ESCOLARIDADE IN ('NÃO INFORMADO') THEN ('Nao Informado')
                END AS DsGrauEscolaridade
                FROM df_pel
                """)

In [0]:
# Cria coluna Id (incremental)

ordem_col = Window.orderBy("SkGrauEscolaridade")

df_grau_escolaridade = df_grau_escolaridade.withColumn("IdGrauEscolaridade",
                                                      row_number().over(ordem_col)) \
                                            .withColumn("DtAtualizacao",
                                                        date_format(current_timestamp(),
                                                                    "dd/MM/yyyy HH:mm:ss"))

In [0]:
# Ordenando colunas na Dimensão
df_grau_escolaridade = df_grau_escolaridade.select("IdGrauEscolaridade", "SkGrauEscolaridade", "DsGrauEscolaridade", "DtAtualizacao")

In [0]:
# Validação do DataFrame
#display(df_grau_escolaridade)

##### Salvar DataFrame da dimensão: Faixa Etária na camada Gold

In [0]:
# Salva DataFrame (dimensão: Faixa Etária) na camada Gold
df_grau_escolaridade.write\
                        .mode('overwrite')\
                        .format('delta')\
                        .option('mergeSchema', 'true')\
                        .save(f'{path_gold_dimensao}/dim_grau_escolaridade')

##### Fim